In [1]:
#Bibliotecas de terceros
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import importlib
from functools import reduce
#Proyecto
import set_paths
from src.colonias import colonias_bj
from src.utils import quitar_acentos
from src.indice_apr import obtener_pm2_colonia, get_indice

## LIMPIAMOS LOS DATOS

In [2]:
dato_actual = pd.read_csv(r"..\datos\properties.csv")[["price", "neighborhood", "built_m2"]]
dato_actual["neighborhood"] = dato_actual["neighborhood"].apply(quitar_acentos)
dato_actual = dato_actual.where(dato_actual["neighborhood"].isin(colonias_bj)).dropna()
dato_actual["price_m2"] = np.round(dato_actual["price"] / dato_actual["built_m2"], 0)
dato_actual.head(10)

,price,neighborhood,built_m2,price_m2
0,680000.0,Del Valle Norte,120.0,5667.0
1,690000.0,Nativitas,210.0,3286.0
2,16000000.0,Letran Valle,511.0,31311.0
3,32000000.0,Del Valle Centro,850.0,37647.0
4,8350000.0,Portales Norte,220.0,37955.0
6,552000.0,San Simon Ticumac,152.0,3632.0
7,552000.0,San Simon Ticumac,1522.0,363.0
8,620000.0,Ermita,320.0,1938.0
9,700000.0,Nativitas,858.0,816.0
10,700000.0,Nativitas,3582.0,195.0


## TOMAMOS LA MEDIANA POR COLONIA

In [3]:
#Tomamos la mediana 
p_m2_median=dato_actual.groupby("neighborhood").median()["price_m2"]
p_m2_26 = pd.DataFrame([p_m2_median.index, p_m2_median.values], index=["colonia", "precio_m2"]).T

In [4]:
p_m2_26.to_csv(r"..\datos\clean\p_m2_26.csv", index=False)

In [5]:
indices_periodos = pd.read_csv(r"..\datos\indice_periodos.csv")
indice_2026 = get_indice(indices_periodos, 1, 2026)
#ratios
ratio_2020 = get_indice(indices_periodos, 1,2020)/indice_2026
ratio_2022 = get_indice(indices_periodos, 1, 2022)/indice_2026
ratio_2024 = get_indice(indices_periodos, 1, 2024)/indice_2026
#df's
p_m2_20 = obtener_pm2_colonia(ratio_2020, r"..\datos\clean\p_m2_26.csv")
p_m2_22 = obtener_pm2_colonia(ratio_2022, r"..\datos\clean\p_m2_26.csv")
p_m2_24 = obtener_pm2_colonia(ratio_2024, r"..\datos\clean\p_m2_26.csv")
periodos_anteriores = [p_m2_20, p_m2_22, p_m2_24]

In [13]:
periodos_anteriores_df = p_m2_20.merge(p_m2_22, on = "colonia").merge(p_m2_24, on = "colonia")
periodos_anteriores_df.columns = ["colonia", "precio_m2_20", "precio_m2_22", "precio_m2_24"]
precios_final = periodos_anteriores_df.merge(p_m2_26, on = "colonia")
precios_final.columns = ["colonia", "precio_m2_20", "precio_m2_22", "precio_m2_24", "precio_m2_26"]

In [15]:
precios_final.to_csv(r"..\datos\clean\precios_final.csv", index=False)

In [ ]:
for i in range(3):
    periodos_anteriores[i].to_csv(rf"..\datos\clean\p_m2_{20+2*i}.csv", index=False)

In [ ]:
precios_final[precios_final["colonia"]=="Postal"]["precio_m2_20"].values[0]

16634.583593054318

In [33]:
precios_final.head()

,colonia,precio_m2_20,precio_m2_22,precio_m2_24,precio_m2_26
0,Acacias,33629.010686,36066.048085,43442.955031,47288.0
1,Actipan,33864.402382,36318.498219,43747.041017,47619.0
2,Alamos,27299.036398,29277.351291,35265.7062,38387.0
3,Americas Unidas,25514.041963,27363.001336,32959.797362,35877.0
4,Ampliacion Napoles,25971.313446,27853.410508,33550.514248,36520.0


In [52]:
test = precios_final
test.iloc[:, 1:] = test.iloc[:, 1:].astype(int)

In [57]:
test["Tam_muestra"] = dato_actual["neighborhood"].value_counts().values

In [ ]:
test.to_csv(r"..\datos\clean\datos_mapa.csv", index=False)